<a href="https://colab.research.google.com/github/Madathanapalleleena/ML_LAB_152/blob/main/ml_lab_ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Comparing Single vs. Ensemble Classifiers

from google.colab import files
uploaded=files.upload()


Saving indian_liver_patient.csv to indian_liver_patient.csv


In [ ]:
import pandas as pd
df=pd.read_csv("indian_liver_patient.csv")
df.head(2)

,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
0,65,Female,0.7,0.1,187,16,18,6.8,3.3,0.90,1
1,62,Male,10.9,5.5,699,64,100,7.5,3.2,0.74,1


In [ ]:
#Single classifier
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

#handling missing and null values
print(df.isnull().sum())

Age                           0
Gender                        0
Total_Bilirubin               0
Direct_Bilirubin              0
Alkaline_Phosphotase          0
Alamine_Aminotransferase      0
Aspartate_Aminotransferase    0
Total_Protiens                0
Albumin                       0
Albumin_and_Globulin_Ratio    4
Dataset                       0
dtype: int64


In [ ]:
df.fillna(df.mean(numeric_only=True), inplace=True)
print(df.isnull().sum())

#converting gender to numerical values - males-1 , females=0
le = LabelEncoder()
df['Gender']=le.fit_transform(df['Gender'])

print('after converting gender to categorical:')
print(df.head(2))

Age                           0
Gender                        0
Total_Bilirubin               0
Direct_Bilirubin              0
Alkaline_Phosphotase          0
Alamine_Aminotransferase      0
Aspartate_Aminotransferase    0
Total_Protiens                0
Albumin                       0
Albumin_and_Globulin_Ratio    0
Dataset                       0
dtype: int64
after converting gender to categorical:
   Age  Gender  Total_Bilirubin  Direct_Bilirubin  Alkaline_Phosphotase  \
0   65       0              0.7               0.1                   187   
1   62       1             10.9               5.5                   699   

   Alamine_Aminotransferase  Aspartate_Aminotransferase  Total_Protiens  \
0                        16                          18             6.8   
1                        64                         100             7.5   

   Albumin  Albumin_and_Globulin_Ratio  Dataset  
0      3.3                        0.90        1  
1      3.2                        0.74     

Has liver disease - 1

does not have liver disease - 0 (2 in actual dataset)

In [ ]:
#Evaluation metrics
def evaluate_model(name, y_true, y_pred):
    print(f"\n{name}")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred):.4f}")
    print(f"F1-score : {f1_score(y_true, y_pred):.4f}")

In [ ]:
X=df.drop('Dataset', axis=1)
y=df['Dataset']
#converting target to binary classes
y=np.where(y==2,1,0) # changes 2--> condition if true 0 else 1
X_train, X_test, y_train, y_test=train_test_split(X,y, test_size=0.3, random_state=42)
#preprocessing
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

evaluate_model("Decision Tree", y_test, y_pred_dt)


Decision Tree
Accuracy : 0.6971
Precision: 0.4423
Recall   : 0.4894
F1-score : 0.4646


In [ ]:
#ensemble classifier
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=5, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
evaluate_model("Random Forest", y_test, y_pred_rf)


Random Forest
Accuracy : 0.7314
Precision: 0.5000
Recall   : 0.3404
F1-score : 0.4051


Accuracy -Random Forest predicts overall more correctly (about 73% vs 69%).

Precision -When RF predicts a positive (disease), it’s more likely correct.

Recall -But RF misses more positive cases (fewer patients correctly identified with disease).

F1 -Because F1 combines precision and recall, when recall drops a lot, F1 also drops.


Random forest classifier is focusing on **majority class** - therefore there is decrease in the value of **recall **

In [ ]:
for n in [1,5,10, 50, 100, 200, 300]:
    rf_temp = RandomForestClassifier(n_estimators=n, random_state=42)
    rf_temp.fit(X_train, y_train)
    y_pred_temp = rf_temp.predict(X_test)
    acc = accuracy_score(y_test, y_pred_temp)
    print(f"Estimators: {n:3d} | Accuracy: {acc:.4f}")
    print(f"Precision: {precision_score(y_test, y_pred_temp ):.4f}")
    print(f"Recall   : {recall_score(y_test, y_pred_temp ):.4f}")
    print(f"F1-score : {f1_score(y_test, y_pred_temp ):.4f}")

Estimators:   1 | Accuracy: 0.6686
Precision: 0.3830
Recall   : 0.3830
F1-score : 0.3830
Estimators:   5 | Accuracy: 0.7029
Precision: 0.4510
Recall   : 0.4894
F1-score : 0.4694
Estimators:  10 | Accuracy: 0.6971
Precision: 0.4118
Recall   : 0.2979
F1-score : 0.3457
Estimators:  50 | Accuracy: 0.7029
Precision: 0.4138
Recall   : 0.2553
F1-score : 0.3158
Estimators: 100 | Accuracy: 0.7314
Precision: 0.5000
Recall   : 0.3404
F1-score : 0.4051
Estimators: 200 | Accuracy: 0.7143
Precision: 0.4615
Recall   : 0.3830
F1-score : 0.4186
Estimators: 300 | Accuracy: 0.7086
Precision: 0.4444
Recall   : 0.3404
F1-score : 0.3855


Recall: “Of all sick people, how many did I catch?” → Focus on actual positives

Precision: “Of the people I flagged as sick, how many truly are sick?” → Focus on predicted positives


**at estimators = 5 it is giving better accuracy and recall**




# why random forest (ensemble) is better?

Random forest works on majority voting therefore more generalized model so better than decision tree - decision tree leads to overfitting when single tree grows in depth

In [ ]:
from sklearn.linear_model import LogisticRegression
dt = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(n_estimators=5, random_state=42)
lr = LogisticRegression(random_state=42, max_iter=1000)

dt.fit(X_train, y_train)
rf.fit(X_train, y_train)
lr.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_pred_rf = rf.predict(X_test)
y_pred_lr = lr.predict(X_test)

#majority voting
predictions = np.array([y_pred_dt, y_pred_rf, y_pred_lr])
y_pred_max = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=predictions)

In [ ]:
#Average Voting (Probabilities)
y_prob_dt = dt.predict_proba(X_test)[:,1]
y_prob_rf = rf.predict_proba(X_test)[:,1]
y_prob_lr = lr.predict_proba(X_test)[:,1]
y_prob_avg = (y_prob_dt + y_prob_rf + y_prob_lr) / 3
y_pred_avg = (y_prob_avg >= 0.5).astype(int)

#Weighted Average Voting
# Assign weights based on model performance (accuracy)
acc_dt = accuracy_score(y_test, y_pred_dt)
acc_rf = accuracy_score(y_test, y_pred_rf)
acc_lr = accuracy_score(y_test, y_pred_lr)

weights = np.array([acc_dt, acc_rf, acc_lr])
y_prob_weighted = (y_prob_dt*weights[0] + y_prob_rf*weights[1] + y_prob_lr*weights[2]) / weights.sum()
y_pred_weighted = (y_prob_weighted >= 0.5).astype(int)

#Evaluation function
def evaluate_model(name, y_true, y_pred):
    print(f"\n{name}")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred):.4f}")
    print(f"F1-score : {f1_score(y_true, y_pred):.4f}")

**Majority voting**

Works with class predictions (not probabilities) from multiple models.
Each base classifier “votes” for a class.
The class with the most votes wins → becomes the final prediction.

**Average Voting**

Take the average of predicted probabilities from all models ,if threshold 0.5 → final prediction

**Weighted average voting**

Take a weighted average of probabilities based on model performance → threshold 0.5 → final prediction



In [ ]:
# Step 12: Evaluate base classifiers
evaluate_model("Decision Tree", y_test, y_pred_dt)
evaluate_model("Random Forest", y_test, y_pred_rf)
evaluate_model("Logistic Regression", y_test, y_pred_lr)

# Step 13: Evaluate ensemble techniques
evaluate_model("Max Voting Ensemble", y_test, y_pred_max)
evaluate_model("Average Voting Ensemble", y_test, y_pred_avg)
evaluate_model("Weighted Average Voting Ensemble", y_test, y_pred_weighted)


Decision Tree
Accuracy : 0.6971
Precision: 0.4423
Recall   : 0.4894
F1-score : 0.4646

Random Forest
Accuracy : 0.7029
Precision: 0.4510
Recall   : 0.4894
F1-score : 0.4694

Logistic Regression
Accuracy : 0.7200
Precision: 0.4500
Recall   : 0.1915
F1-score : 0.2687

Max Voting Ensemble
Accuracy : 0.6914
Precision: 0.4000
Recall   : 0.2979
F1-score : 0.3415

Average Voting Ensemble
Accuracy : 0.7200
Precision: 0.4773
Recall   : 0.4468
F1-score : 0.4615

Weighted Average Voting Ensemble
Accuracy : 0.7200
Precision: 0.4773
Recall   : 0.4468
F1-score : 0.4615


**Weighted Average Voting** is giving more accuracy compared to other ensemble techniques

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.neighbors import KNeighborsClassifier

dt = DecisionTreeClassifier(random_state=42)
lr = LogisticRegression(random_state=42, max_iter=1000)
knn = KNeighborsClassifier(n_neighbors=5)

In [ ]:
hard_voting_clf = VotingClassifier(
    estimators=[('dt', dt), ('lr', lr), ('knn', knn)],
    voting='hard'
)

#Define Soft Voting Classifier
soft_voting_clf = VotingClassifier(
    estimators=[('dt', dt), ('lr', lr), ('knn', knn)],
    voting='soft'  # uses predicted probabilities
)

#Train classifiers
hard_voting_clf.fit(X_train, y_train)
soft_voting_clf.fit(X_train, y_train)

#Make predictions
y_pred_hard = hard_voting_clf.predict(X_test)
y_pred_soft = soft_voting_clf.predict(X_test)

#Evaluation function
def evaluate_model(name, y_true, y_pred):
    print(f"\n=== {name} ===")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred):.4f}")
    print(f"F1-score : {f1_score(y_true, y_pred):.4f}")

# Step 14: Evaluate both voting classifiers
evaluate_model("Hard Voting Classifier", y_test, y_pred_hard)
evaluate_model("Soft Voting Classifier", y_test, y_pred_soft)


=== Hard Voting Classifier ===
Accuracy : 0.6914
Precision: 0.4000
Recall   : 0.2979
F1-score : 0.3415

=== Soft Voting Classifier ===
Accuracy : 0.7029
Precision: 0.4419
Recall   : 0.4043
F1-score : 0.4222


**Hard voting and max voting** are giving same results
soft voting -- probability based voting -- giving better results

probability based are giving better results